# 03 — Linear probing → initial gene predictions (`ŷ_init`)

GeneRAG's biological retrieval query is the initial gene prediction `ŷ_init` produced
by a frozen foundation-model encoder plus a small head. The paper calls this the
*decoder* `D`; in practice the standard pathology-foundation-model evaluation protocol
uses a **linear probe** — encoder fully frozen, single `nn.Linear` head trained on the
training-set ground-truth expression. This notebook reproduces that step.

What this produces

* For every (backbone, anchor-gene-list) combination, a `.pt` file shaped
  `(n_test_spots * num_rep, n_anchor_genes)` saved into a directory that
  `examples/run_experiment.py` (and the new `generag.data.load_test_predictions`)
  can ingest directly.

Prerequisites

* `02_build_bank.ipynb` produced `1spot_<backbone>_ebd_aug/*.pt` and the anchor gene list.
* The corresponding `.h5ad` slides are available under `<organ>/st/`.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import anndata
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.sparse import issparse
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
# Cohort + split — keep in sync with the previous notebooks.
ORGAN = 'PRAD'
DATA_ROOT = './hest1k_datasets'
data_path = os.path.join(DATA_ROOT, ORGAN)
st_path = os.path.join(data_path, 'st')
processed_dir = os.path.join(data_path, 'processed_data')

TEST_SLIDE = 'MEND145'
TRAIN_SLIDES = [f'MEND{i}' for i in range(139, 163) if i != 155 and f'MEND{i}' != TEST_SLIDE]

# Anchor-gene list (file produced by notebook 02). You may replace this with any
# panel — top-k HVGs, marker genes, hub genes, etc.
GENE_LIST_FILE = os.path.join(processed_dir, 'selected_gene_list.txt')
GENE_BASENAME = os.path.splitext(os.path.basename(GENE_LIST_FILE))[0].replace('selected_', '').replace('_list', '')

# Output: directory consumed by examples/run_experiment.py
LR_PRED_PT_DIR = os.path.join(data_path, 'init_pred_fm_pt')
os.makedirs(LR_PRED_PT_DIR, exist_ok=True)

## 1) Load per-spot embeddings (test and train sides)

In [ ]:
def load_slide_embedding(embed_dir: str, slide_id: str, suffix: str) -> np.ndarray | None:
    """Load one slide's `.pt` and average over the augmentation axis if present."""
    fpath = os.path.join(embed_dir, slide_id + suffix)
    if not os.path.exists(fpath):
        return None
    emb = torch.load(fpath, map_location='cpu')
    arr = emb.cpu().numpy() if isinstance(emb, torch.Tensor) else np.asarray(emb)
    if arr.ndim == 3:
        arr = arr.mean(axis=1)   # (n_spots, n_aug, d) -> (n_spots, d)
    return arr


def load_train_embeddings(embed_dir: str, slides: list[str], suffix: str) -> np.ndarray:
    parts = [load_slide_embedding(embed_dir, sid, suffix) for sid in slides]
    return np.vstack([p for p in parts if p is not None])

In [ ]:
# Backbones to evaluate. Add/remove entries as you produce more embeddings in notebook 02.
BACKBONES = [
    {'name': 'UNI',    'dir': os.path.join(processed_dir, '1spot_uni_ebd_aug'),    'suffix': '_uni_aug.pt'},
    # {'name': 'Exaone', 'dir': os.path.join(processed_dir, '1spot_exaone_ebd_aug'), 'suffix': '_exaone_aug.pt'},
]

for bb in BACKBONES:
    bb['test']  = load_slide_embedding(bb['dir'], TEST_SLIDE, bb['suffix'])
    bb['train'] = load_train_embeddings(bb['dir'], TRAIN_SLIDES, bb['suffix'])
    print(f"{bb['name']}: train {bb['train'].shape}, test {None if bb['test'] is None else bb['test'].shape}")

## 2) Load anchor-gene expression (regression target)

Restricted to the gene panel from notebook 02 and the genes that are actually present
in both the test slide and every train slide.

In [ ]:
def to_dense(X):
    return X.toarray() if issparse(X) else np.asarray(X)


selected_genes = [g.strip() for g in open(GENE_LIST_FILE) if g.strip()]
adata_first = anndata.read_h5ad(os.path.join(st_path, TRAIN_SLIDES[0] + '.h5ad'))
adata_test  = anndata.read_h5ad(os.path.join(st_path, TEST_SLIDE + '.h5ad'))
common = set(adata_first.var_names) & set(adata_test.var_names)
genes_use = [g for g in selected_genes if g in common and g in adata_first.var_names]
print(f'anchor genes effectively used: {len(genes_use)} / {len(selected_genes)}')

expr_test = to_dense(adata_test[:, genes_use].X).astype(np.float32)


def _load_expr_slide(sid: str) -> tuple[str, np.ndarray]:
    adata = anndata.read_h5ad(os.path.join(st_path, sid + '.h5ad'))
    return sid, to_dense(adata[:, genes_use].X)

slide_to_expr: dict[str, np.ndarray] = {}
with ThreadPoolExecutor(max_workers=8) as ex:
    for fut in tqdm(as_completed([ex.submit(_load_expr_slide, s) for s in TRAIN_SLIDES]),
                    total=len(TRAIN_SLIDES), desc='loading train expr'):
        sid, X = fut.result()
        slide_to_expr[sid] = X

expr_train = np.vstack([slide_to_expr[s] for s in TRAIN_SLIDES]).astype(np.float32)
print('expr_train:', expr_train.shape, '| expr_test:', expr_test.shape)

## 3) Train a linear probe per backbone

Encoder frozen, head = `nn.Linear(d, G)`, Adam (lr 1e-3, wd 1e-5), batch 4096, 100
epochs, MSE loss. Features are standardised with a `StandardScaler` fitted on the
training spots.

In [ ]:
def train_linear_probe(
    emb_train: np.ndarray,
    emb_test: np.ndarray,
    Y_train: np.ndarray,
    *,
    lr: float = 1e-3,
    weight_decay: float = 1e-5,
    batch_size: int = 4096,
    n_epochs: int = 100,
) -> np.ndarray:
    """Train a single linear layer and return test-time predictions."""
    scaler = StandardScaler().fit(emb_train.astype(np.float32))
    Xtr = torch.from_numpy(scaler.transform(emb_train).astype(np.float32)).to(DEVICE)
    Xte = torch.from_numpy(scaler.transform(emb_test).astype(np.float32)).to(DEVICE)
    Ytr = torch.from_numpy(Y_train).to(DEVICE)

    head = torch.nn.Linear(Xtr.shape[1], Ytr.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=weight_decay)

    head.train()
    for _ in range(n_epochs):
        perm = torch.randperm(len(Xtr), device=DEVICE)
        for i in range(0, len(Xtr), batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = torch.nn.functional.mse_loss(head(Xtr[idx]), Ytr[idx])
            loss.backward()
            opt.step()

    head.eval()
    with torch.no_grad():
        return head(Xte).cpu().numpy()

## 4) Save predictions in the format `run_experiment.py` expects

The downstream loader (`generag.data.load_test_predictions`) reads a `.pt` tensor of
shape `(n_spots * num_rep, n_anchor_genes)` and averages over `num_rep` to obtain the
per-spot anchor query `y_anchor`. We use `num_rep=20` for compatibility with the
paper's existing checkpoints; for a deterministic linear-probe output every replicate
is identical so the average is the prediction itself.

In [ ]:
def save_predictions_pt(
    y_pred: np.ndarray,
    output_path: str,
    num_rep: int = 20,
) -> None:
    """Repeat each spot ``num_rep`` times and save as ``.pt``."""
    expanded = np.repeat(y_pred.astype(np.float32), num_rep, axis=0)  # (n*num_rep, G)
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    torch.save(torch.from_numpy(expanded), output_path)
    print('wrote', output_path, expanded.shape)

In [ ]:
for bb in BACKBONES:
    if bb['test'] is None or bb['train'] is None:
        print(f'skipping {bb["name"]}: missing embeddings')
        continue
    if bb['train'].shape[0] != expr_train.shape[0] or bb['test'].shape[0] != expr_test.shape[0]:
        print(f'skipping {bb["name"]}: row count mismatch '
              f'(train {bb["train"].shape[0]} vs {expr_train.shape[0]}, '
              f'test {bb["test"].shape[0]} vs {expr_test.shape[0]})')
        continue

    y_pred = train_linear_probe(bb['train'], bb['test'], expr_train)
    out_pt = os.path.join(LR_PRED_PT_DIR, f"generated_samples_lr_{bb['name']}_{GENE_BASENAME}_20sample.pt")
    save_predictions_pt(y_pred, out_pt, num_rep=20)

    # Print a quick per-gene Pearson r summary so you can sanity-check the probe.
    from scipy.stats import pearsonr
    corrs = np.array([
        pearsonr(expr_test[:, j], y_pred[:, j])[0] if np.std(expr_test[:, j]) > 0 else np.nan
        for j in range(expr_test.shape[1])
    ])
    valid = np.isfinite(corrs)
    print(f'{bb["name"]} per-gene Pearson r: mean={np.nanmean(corrs[valid]):.4f}, '
          f'top-10={np.nanmean(np.sort(corrs[valid])[-10:]):.4f}, '
          f'top-50={np.nanmean(np.sort(corrs[valid])[-50:]):.4f}')

## What you should see now

```
PRAD/init_pred_fm_pt/
└── generated_samples_lr_<BACKBONE>_<anchor-list>_20sample.pt
```

Each of those files is one `(backbone, anchor-gene-list)` combination, ready to be
picked up by `examples/run_experiment.py` (via `discover_tasks`). At that point the
GeneRAG retrieval pipeline can run end-to-end:

```bash
python examples/run_experiment.py
```

For a quick smoke test on a single configuration use the `ONLY_MODELS` / `ONLY_GENES`
env vars supported by that script.